# Sold Week 6

In [15]:
import pandas as pd

sold = pd.read_csv("../../IDX_Data/sold_cleaned_week4_5.csv")
print("Shape:", sold.shape)
sold.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_71908/2464178534.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("../../IDX_Data/sold_cleaned_week4_5.csv")


Shape: (140752, 76)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
0,1249888.0,1073295259,sandra.mcfetridge@gmail.com,2024-05-01,1262555.0,Sandra,McFetridge,33.477414,-117.664879,33450 Paseo El Lazo,...,92675,Compass,395.0,NaN,33450 Paseo El Lazo,2024-05,7.0600,False,False,False
1,1025000.0,1073169832,doug@socalmodern.com,2024-05-02,962500.0,Douglas,Kramer,33.812886,-118.097004,3108 Kallin Avenue,...,90808,Re/Max R. E. Specialists,0.0,5256.0,3108 Kallin Avenue,2024-05,7.0600,False,False,False
2,725000.0,1073164974,zachmickelson1@gmail.com,2024-04-30,725000.0,Zachary,Mickelson,33.740706,-117.888816,505 S Spruce Street,...,92703,Luxury Homes of California,0.0,6400.0,505 S Spruce Street,2024-04,6.9925,False,False,False
3,730000.0,1071104851,Antonio@AntonioMatier.com,2024-04-09,730000.0,Antonio,Matier,NaN,NaN,2456 Havenscourt Blvd,...,94605,BHHS Drysdale Properties,NaN,4000.0,2456 Havenscourt Blvd,2024-04,6.9925,False,False,False
4,1324000.0,1069630806,amysaflar@gmail.com,2024-04-08,1291000.0,Amy,Saflar,32.846579,-117.221633,3011 Pennant Way,...,92122,Alan J Yaghdjian,0.0,12300.0,3011 Pennant Way,2024-04,6.9925,False,False,False


In [16]:
date_cols = [
    'CloseDate',
    'PurchaseContractDate',
    'ListingContractDate'
]

for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')

In [17]:
# Price ratio
sold['price_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']
# Price per sq ft
sold['price_per_sqft'] = sold['ClosePrice'] / sold['LivingArea']

In [18]:
# Year-month
sold['year_month'] = sold['CloseDate'].dt.to_period('M')

# Listing → Contract
sold['listing_to_contract_days'] = (
    sold['PurchaseContractDate'] - sold['ListingContractDate']
).dt.days

# Contract → Close
sold['contract_to_close_days'] = (
    sold['CloseDate'] - sold['PurchaseContractDate']
).dt.days

In [19]:
sold[[
    'ClosePrice',
    'OriginalListPrice',
    'price_ratio',
    'price_per_sqft',
    'listing_to_contract_days',
    'contract_to_close_days',
    'year_month'
]].head()

,ClosePrice,OriginalListPrice,price_ratio,price_per_sqft,listing_to_contract_days,contract_to_close_days,year_month
0,1262555.0,1249888.0,1.010135,664.852554,20.0,73.0,2024-05
1,962500.0,1025000.0,0.939024,851.769912,104.0,2.0,2024-05
2,725000.0,725000.0,1.000000,580.000000,18.0,81.0,2024-04
3,730000.0,730000.0,1.000000,576.619273,35.0,58.0,2024-04
4,1291000.0,1324000.0,0.975076,716.426193,51.0,17.0,2024-04


In [20]:
# Remove unrealistic or invalid time values
sold = sold[
    (sold['listing_to_contract_days'] >= 0) &
    (sold['contract_to_close_days'] >= 0)
]

In [21]:
import numpy as np

# Fix infinite values in price_ratio
sold['price_ratio'] = sold['price_ratio'].replace([np.inf, -np.inf], np.nan)
sold = sold[sold['OriginalListPrice'] > 0]

In [22]:
county_summary = sold.groupby('CountyOrParish').agg({
    'ClosePrice': ['mean', 'median'],
    'price_ratio': 'mean',
    'DaysOnMarket': 'mean'
}).reset_index()

county_summary.head()

CountyOrParish    ClosePrice            price_ratio DaysOnMarket
                          mean     median        mean         mean
0        Alameda  1.309755e+06  1180000.0    1.205642    18.253149
1         Amador  3.529167e+05   380000.0    0.854278    54.000000
2          Butte  4.240750e+05   401000.0    1.766977    26.156495
3      Calaveras  5.546107e+05   475000.0    0.959099    28.047619
4          Clark  3.750000e+05   375000.0    0.974026     2.000000

In [23]:
# --- Clean invalid price_ratio values ---

# Step 1: Remove bad OriginalListPrice values (like 0 or 1)
before_rows = sold.shape[0]

sold = sold[sold['OriginalListPrice'] > 10000]

# Step 2: Remove extreme/unrealistic price ratios
sold = sold[sold['price_ratio'] < 5]

after_rows = sold.shape[0]

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", after_rows)

# Step 3: Check distribution again
print("\nUpdated price_ratio summary:")
print(sold['price_ratio'].describe())

Rows before cleaning: 139694
Rows after cleaning: 139503

Updated price_ratio summary:
count    139503.000000
mean          1.002693
std           0.090685
min           0.000667
25%           0.967185
50%           1.000000
75%           1.027451
max           4.000000
Name: price_ratio, dtype: float64


In [24]:
county_summary.columns = [
    'CountyOrParish',
    'mean_price',
    'median_price',
    'avg_price_ratio',
    'avg_days_on_market'
]

county_summary.head()

,CountyOrParish,mean_price,median_price,avg_price_ratio,avg_days_on_market
0,Alameda,1.309755e+06,1180000.0,1.205642,18.253149
1,Amador,3.529167e+05,380000.0,0.854278,54.000000
2,Butte,4.240750e+05,401000.0,1.766977,26.156495
3,Calaveras,5.546107e+05,475000.0,0.959099,28.047619
4,Clark,3.750000e+05,375000.0,0.974026,2.000000


In [25]:
type_summary = sold.groupby('PropertyType').agg({
    'ClosePrice': 'median',
    'price_ratio': 'mean',
    'DaysOnMarket': 'mean'
}).reset_index()

type_summary

,PropertyType,ClosePrice,price_ratio,DaysOnMarket
0,Residential,850000.0,1.002693,23.544124


In [26]:
print("Any inf in price_ratio:", np.isinf(sold['price_ratio']).sum()) 

Any inf in price_ratio: 0


In [27]:
sold.sort_values('price_ratio', ascending=False)[[
    'ClosePrice',
    'OriginalListPrice',
    'price_ratio'
]].head(10)

,ClosePrice,OriginalListPrice,price_ratio
52347,2820000.0,705000.0,4.000000
52350,2820000.0,705000.0,4.000000
52349,2820000.0,705000.0,4.000000
52348,2820000.0,705000.0,4.000000
10356,130000.0,35000.0,3.714286
91202,367000.0,100000.0,3.670000
53263,753000.0,250000.0,3.012000
89042,2518000.0,888520.0,2.833926
58022,1500000.0,600000.0,2.500000
18280,121000.0,50000.0,2.420000


In [28]:
sold.to_csv("sold_week6_feature_engineered.csv", index=False)
print("Week 6 dataset saved.")

Week 6 dataset saved.


### Week 6 Summary

- Created key price and time features  
- Cleaned and validated new columns  
- Removed invalid values  
- Verified outputs with sample data  
- Generated grouped summaries for analysis  
- Prepared dataset for dashboard use